# ChurnGuard AI — Data Preparation
Loads the cleaned dataset (from EDA), engineers features, encodes
categoricals with a REUSABLE, SAVED encoder, and produces train/test
splits ready for modeling.

Why a saved encoder matters: later, the API will receive one new
customer at a time and must transform it into EXACTLY the same columns,
in the same order, that the model was trained on. Recomputing
`pd.get_dummies` from scratch on a single row can't do that reliably
(a single customer won't contain every category, so columns would be
missing or in a different order). Fitting a `OneHotEncoder` once here
and saving it solves that.

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

RAW_PATH = "../data/processed/telco_churn_cleaned.csv"
OUTPUT_DIR = "../data/processed"
MODELS_DIR = "../models"
RANDOM_STATE = 42

## 1. Load the cleaned dataset (output of the EDA notebook)

In [ ]:
df = pd.read_csv(RAW_PATH)
print("Loaded:", df.shape)
df.head()

## 2. Feature engineering
- `tenure_bucket`: groups customers by loyalty stage
- `num_services`: count of add-on services subscribed
- `avg_charge_per_service`: TotalCharges spread across services
- `is_month_to_month`: explicit flag for the strongest expected churn driver
- `has_internet`: segment with no internet service at all

In [ ]:
# Safety net: ensure TotalCharges is numeric even if an unclean CSV is passed in
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)

df["tenure_bucket"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, 48, np.inf],
    labels=["0-12m", "12-24m", "24-48m", "48m+"],
)

service_cols = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies",
]
df["num_services"] = (df[service_cols] == "Yes").sum(axis=1)
df["avg_charge_per_service"] = df["TotalCharges"] / df["num_services"].replace(0, 1)
df["is_month_to_month"] = (df["Contract"] == "Month-to-month").astype(int)
df["has_internet"] = (df["InternetService"] != "No").astype(int)

print("After feature engineering:", df.shape)

## 3. Encoding — with a saved, reusable encoder
- Target (`Churn`) extracted and mapped to 0/1
- `customerID` dropped from features (pure identifier, no predictive value)
- Binary Yes/No columns + `gender` mapped to 0/1 (simple, deterministic —
  no fitting needed, so no leakage risk either way)
- Remaining multi-category columns encoded with `OneHotEncoder`, FIT ONLY
  on the training data, then reused to transform the test data and later
  any new customer coming through the API.
  `handle_unknown="ignore"` means a category never seen in training
  (e.g. a new payment method added later) won't crash the pipeline —
  it just gets encoded as all-zeros instead of raising an error.

In [ ]:
target = df["Churn"].map({"Yes": 1, "No": 0})
features = df.drop(columns=["Churn", "customerID"], errors="ignore")

binary_cols = [c for c in features.columns if set(features[c].dropna().unique()) <= {"Yes", "No"}]
for col in binary_cols:
    features[col] = features[col].map({"Yes": 1, "No": 0})

if "gender" in features.columns:
    features["gender"] = features["gender"].map({"Male": 1, "Female": 0})

categorical_cols = features.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = [c for c in features.columns if c not in categorical_cols]
print("One-hot encoding:", categorical_cols)

X = features
y = target

## 4. Train/test split
Split BEFORE fitting the encoder and BEFORE any class balancing — both
must only ever "see" the training data, or the test set stops being an
honest measure of real-world performance (this is the same leakage rule
as with SMOTE, applied to the encoder too). `stratify=y` keeps the real
~26.5% churn rate identical in both splits.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train set: {X_train.shape}, Test set: {X_test.shape}")

## 5. Fit the encoder on TRAIN ONLY, then transform both splits

In [ ]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first")
encoder.fit(X_train[categorical_cols])

encoded_cols = encoder.get_feature_names_out(categorical_cols)

def apply_encoding(X_part):
    encoded = pd.DataFrame(
        encoder.transform(X_part[categorical_cols]),
        columns=encoded_cols,
        index=X_part.index,
    )
    numeric_part = X_part[numeric_cols].reset_index(drop=True)
    encoded = encoded.reset_index(drop=True)
    return pd.concat([numeric_part, encoded], axis=1)

X_train_encoded = apply_encoding(X_train)
X_test_encoded = apply_encoding(X_test)

print("After encoding — train:", X_train_encoded.shape, " test:", X_test_encoded.shape)
print(f"Train churn rate: {y_train.mean():.1%}, Test churn rate: {y_test.mean():.1%}")

## 6. Save everything the next stages need
- Train/test splits, ready for `train_churnguard.ipynb`
- The fitted encoder (`encoder.joblib`) — the API loads this to transform
  a new customer exactly the same way, instead of re-implementing the logic
- `feature_columns.joblib` — the exact column order the model expects.
  Any dataframe fed to the model later (API, batch scoring) must be
  reindexed to this column list before predicting.

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

X_train_encoded.to_csv(os.path.join(OUTPUT_DIR, "X_train.csv"), index=False)
X_test_encoded.to_csv(os.path.join(OUTPUT_DIR, "X_test.csv"), index=False)
y_train.to_csv(os.path.join(OUTPUT_DIR, "y_train.csv"), index=False)
y_test.to_csv(os.path.join(OUTPUT_DIR, "y_test.csv"), index=False)

joblib.dump(encoder, os.path.join(MODELS_DIR, "encoder.joblib"))
joblib.dump(list(X_train_encoded.columns), os.path.join(MODELS_DIR, "feature_columns.joblib"))
joblib.dump(categorical_cols, os.path.join(MODELS_DIR, "categorical_cols.joblib"))
joblib.dump(numeric_cols, os.path.join(MODELS_DIR, "numeric_cols.joblib"))

print("Saved train/test splits to", os.path.abspath(OUTPUT_DIR))
print("Saved encoder + feature metadata to", os.path.abspath(MODELS_DIR))